# Kerachut beach: satellite image check (Sentinel-2 and Landsat)

Project: set `PROJECT` in the first code cell (your own Google Cloud project ID)

What this notebook does:
1. Connects to Earth Engine.
2. Lets you confirm or redraw the study box over Pantai Kerachut.
3. Builds a scene inventory for Sentinel-2 (10 m, 2017 onward) and Landsat 5/7/8/9 (30 m, 1987 onward), with the percentage of cloud-free pixels over the beach for every scene.
4. Picks the clearest scene per year and shows them side by side.
5. Shows a swipe map (earliest vs latest clear Sentinel-2) and an NDWI water index panel.
6. Optionally exports the chosen scenes to your Google Drive as GeoTIFFs.

Run the cells in order with Shift+Enter.

In [ ]:
# 1. Setup (Colab already has earthengine-api; geemap may need installing)
!pip install -q geemap

import ee, geemap
import pandas as pd
import matplotlib.pyplot as plt
import requests, io
from PIL import Image

from google.colab import output
output.enable_custom_widget_manager()   # needed for interactive maps in Colab

# >>> Replace with YOUR Google Cloud project ID (registered for Earth Engine).
# >>> See docs/GEE_tutorial.md, section 3, for how to find it.
PROJECT = 'YOUR-GEE-PROJECT-ID'
ee.Authenticate()          # a sign-in prompt appears the first time
ee.Initialize(project=PROJECT)
print('Earth Engine ready:', PROJECT)

## 2. Study area

The centre point below is approximate. On the map:
* Check that the red box covers the whole Kerachut beach plus some sea and the vegetation behind it.
* If it is off, use the rectangle tool on the left of the map to draw your own box, then run the next cell. Your drawn box replaces the default.

In [ ]:
CENTER_LAT = 5.4510      # approximate, please verify on the map
CENTER_LON = 100.1840
HALF_SIZE  = 0.008       # degrees, about 900 m each way

default_roi = ee.Geometry.Rectangle([CENTER_LON - HALF_SIZE, CENTER_LAT - HALF_SIZE,
                                     CENTER_LON + HALF_SIZE, CENTER_LAT + HALF_SIZE])

m = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=15)
m.add_basemap('SATELLITE')
m.addLayer(ee.FeatureCollection([ee.Feature(default_roi)]).style(color='red', fillColor='00000000', width=2),
           {}, 'Default study box')
m

In [ ]:
# Use the drawn rectangle if there is one, otherwise the default box
r = m.user_roi
if r is not None:
    roi = r.geometry() if isinstance(r, ee.Feature) else ee.Geometry(r)
    print('Using your drawn box')
else:
    roi = default_roi
    print('Using the default box')

c = roi.centroid(1).coordinates().getInfo()
print('Box centre (lon, lat):', [round(v, 5) for v in c])
print('Box area (ha):', round(roi.area(1).getInfo() / 1e4, 1))

## 3. Scene inventory

`clear_frac` is the share of pixels inside the box that are cloud and shadow free.
* Sentinel-2 uses its own Scene Classification (SCL) band.
* Landsat uses the QA_PIXEL bits (fill, dilated cloud, cirrus, cloud, shadow).

Images are kept unmasked so the thumbnails look normal; the mask is only used to score them.
Note: Landsat 7 scenes after May 2003 have stripes (SLC-off). They are flagged in the table.

In [ ]:
START, END = '1987-01-01', '2026-12-31'
BANDS = ['blue', 'green', 'red', 'nir', 'swir1']

# ---- Sentinel-2 surface reflectance, cloud mask from the built-in SCL band ----
# SCL classes treated as not clear: 0 no data, 1 saturated, 3 cloud shadow,
# 8 cloud medium, 9 cloud high, 10 cirrus
BAD_SCL = [0, 1, 3, 8, 9, 10]

def prep_s2(img):
    scl = img.select('SCL')
    clear = scl.remap(BAD_SCL, [0] * len(BAD_SCL), 1).rename('clear')
    frac = clear.reduceRegion(ee.Reducer.mean(), roi, 10, maxPixels=1e8).get('clear')
    sr = img.select(['B2', 'B3', 'B4', 'B8', 'B11'], BANDS).divide(10000)
    return sr.copyProperties(img, ['system:time_start']).set({
        'clear_frac': frac, 'sensor': 'S2', 'res_m': 10,
        'img_id': ee.String('S2_').cat(img.get('system:index'))})

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(roi).filterDate(START, END)
      .map(prep_s2))

# ---- Landsat Collection 2 Level-2 ----
def make_ls_prep(sensor, src_bands):
    def prep(img):
        qa = img.select('QA_PIXEL')
        clear = qa.bitwiseAnd(0b11111).eq(0).rename('clear')
        frac = clear.reduceRegion(ee.Reducer.mean(), roi, 30, maxPixels=1e8).get('clear')
        sr = img.select(src_bands, BANDS).multiply(0.0000275).add(-0.2)
        return sr.copyProperties(img, ['system:time_start']).set({
            'clear_frac': frac, 'sensor': sensor, 'res_m': 30,
            'img_id': ee.String(sensor + '_').cat(img.get('system:index'))})
    return prep

L57 = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5']
L89 = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6']
def ls(cid, sensor, bands):
    return ee.ImageCollection(cid).filterBounds(roi).filterDate(START, END).map(make_ls_prep(sensor, bands))

landsat = (ls('LANDSAT/LT05/C02/T1_L2', 'L5', L57)
           .merge(ls('LANDSAT/LE07/C02/T1_L2', 'L7', L57))
           .merge(ls('LANDSAT/LC08/C02/T1_L2', 'L8', L89))
           .merge(ls('LANDSAT/LC09/C02/T1_L2', 'L9', L89)))

allimg = s2.merge(landsat).filter(ee.Filter.notNull(['clear_frac']))

def inventory(col):
    props = ['img_id', 'sensor', 'res_m', 'clear_frac']
    d = {p: col.aggregate_array(p).getInfo() for p in props}
    d['date'] = col.aggregate_array('system:time_start').getInfo()
    df = pd.DataFrame(d)
    df['date'] = pd.to_datetime(df['date'], unit='ms')
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['slc_off'] = (df['sensor'] == 'L7') & (df['date'] > '2003-05-31')
    return df.sort_values('date').reset_index(drop=True)

print('Building inventory (this can take a minute)...')
inv = inventory(allimg)
inv.to_csv('kerachut_scene_inventory.csv', index=False)
print(len(inv), 'scenes in total')
print(inv.groupby('sensor').agg(scenes=('img_id', 'size'),
                                 first=('date', 'min'), last=('date', 'max'),
                                 clear_95=('clear_frac', lambda s: (s >= 0.95).sum())))

In [ ]:
# How many usable (>= 95% clear) scenes per year, by sensor
MIN_CLEAR = 0.95
good = inv[(inv.clear_frac >= MIN_CLEAR) & (~inv.slc_off)]
counts = good.pivot_table(index='year', columns='sensor', values='img_id', aggfunc='count').fillna(0).astype(int)
ax = counts.plot(kind='bar', stacked=True, figsize=(12, 3.5))
ax.set_ylabel('Clear scenes'); ax.set_title(f'Scenes with >= {int(MIN_CLEAR*100)}% clear pixels over Kerachut')
plt.tight_layout(); plt.show()

# Monsoon check: which months give clear views
mon = good.groupby('month').size().reindex(range(1, 13), fill_value=0)
print('Clear scenes by month (all years):')
print(mon.to_string())

## 4. Clearest scene per year, side by side

For each year the notebook prefers Sentinel-2 (10 m) when available, otherwise Landsat 8/9, then Landsat 5. Landsat 7 SLC-off scenes are skipped.

Tide is not controlled here. The waterline will shift with tide, so for erosion use the vegetation line and berm edge as your main reference, or later add tide correction.

In [ ]:
PRIORITY = {'S2': 0, 'L9': 1, 'L8': 1, 'L5': 2, 'L7': 3}
pick = (good.assign(pri=good.sensor.map(PRIORITY))
            .sort_values(['year', 'pri', 'clear_frac'], ascending=[True, True, False])
            .groupby('year').head(1).reset_index(drop=True))
print(pick[['year', 'date', 'sensor', 'clear_frac', 'img_id']].to_string(index=False))

VIS = {'bands': ['red', 'green', 'blue'], 'min': 0.0, 'max': 0.25, 'gamma': 1.3}

def get_img(img_id):
    return ee.Image(allimg.filter(ee.Filter.eq('img_id', img_id)).first())

def thumb(img_id, px=400):
    url = get_img(img_id).getThumbURL({**VIS, 'region': roi, 'dimensions': px, 'format': 'png'})
    return Image.open(io.BytesIO(requests.get(url).content))

n = len(pick)
cols = 5
rows = -(-n // cols)
fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.4 * rows))
axes = axes.ravel() if n > 1 else [axes]
for ax in axes: ax.axis('off')
for ax, (_, row) in zip(axes, pick.iterrows()):
    ax.imshow(thumb(row.img_id))
    ax.set_title(f"{row.date:%Y-%m-%d}  {row.sensor} ({row.res_m} m)", fontsize=9)
plt.suptitle('Pantai Kerachut, clearest scene per year', y=1.0)
plt.tight_layout(); plt.show()

## 5. Swipe map and water index

The swipe map compares the earliest and latest clear Sentinel-2 scenes on the same basemap. Drag the slider.

NDWI = (green - nir) / (green + nir). Water is positive, sand and vegetation negative. This is the band that shoreline extraction will use later.

In [ ]:
s2_pick = good[good.sensor == 'S2'].sort_values('date')
first_id, last_id = s2_pick.iloc[0].img_id, s2_pick.iloc[-1].img_id
d1, d2 = s2_pick.iloc[0].date.date(), s2_pick.iloc[-1].date.date()

m2 = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=16)
left  = geemap.ee_tile_layer(get_img(first_id), VIS, f'S2 {d1}')
right = geemap.ee_tile_layer(get_img(last_id),  VIS, f'S2 {d2}')
m2.split_map(left, right)
print(f'Left: {d1}   Right: {d2}')
m2

In [ ]:
def ndwi(img):
    return img.normalizedDifference(['green', 'nir']).rename('ndwi')

NDWI_VIS = {'min': -0.5, 'max': 0.5, 'palette': ['8c510a', 'f6e8c3', 'f5f5f5', '80cdc1', '01665e']}
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, (img_id, d) in zip(axes[[0, 2]], [(first_id, d1), (last_id, d2)]):
    ax.imshow(thumb(img_id, 500)); ax.set_title(f'True colour {d}'); ax.axis('off')
for ax, (img_id, d) in zip(axes[[1, 3]], [(first_id, d1), (last_id, d2)]):
    url = ndwi(get_img(img_id)).getThumbURL({**NDWI_VIS, 'region': roi, 'dimensions': 500, 'format': 'png'})
    ax.imshow(Image.open(io.BytesIO(requests.get(url).content))); ax.set_title(f'NDWI {d}'); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Optional: export the chosen scenes to Google Drive

Set `EXPORT = True` and run. Each scene goes to a Drive folder called `Kerachut_GEE` as a 5-band GeoTIFF (blue, green, red, nir, swir1) at its native resolution, in UTM zone 47N (EPSG:32647). Progress is shown on the Earth Engine Tasks page.

Also download `kerachut_scene_inventory.csv` from the Colab file panel (folder icon on the left) to keep the full scene list.

In [ ]:
EXPORT = False

if EXPORT:
    for _, row in pick.iterrows():
        task = ee.batch.Export.image.toDrive(
            image=get_img(row.img_id).select(BANDS).toFloat(),
            description=f"Kerachut_{row.date:%Y%m%d}_{row.sensor}",
            folder='Kerachut_GEE',
            region=roi, scale=int(row.res_m), crs='EPSG:32647', maxPixels=1e9)
        task.start()
        print('Started', task.config['description'])
else:
    print('Export is off. Set EXPORT = True to send the scenes to Drive.')